In [ ]:
import scanpy as sc
import anndata as ad
import scvi
import sys
sys.path.append("/multiHIVE/src")
from multiHIVE import multiHIVE
import pandas as pd
import numpy as np

In [2]:
import torch, random
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)

In [ ]:
adata = sc.read('/Data/RNA_ADT/Hao/pbmc_seurat_v4.h5ad')

In [ ]:
sc.pp.normalize_total(adata ,target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat_v3",
    batch_key="batch",
    subset=True,
    layer="counts"
)
hvg = 2000
multiHIVE.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
    protein_expression_obsm_key="protein_counts"
)

In [ ]:
vae = multiHIVE(adata, latent_distribution="normal", kl_dot_product=True, deep_network=True,
               n_genes=adata.shape[1],
    n_regions=0,
    n_proteins=228)
vae.train(max_epochs = 200,  early_stopping =False)


vae.get_latent_representation()

generated_data = vae.posterior_predictive_sample(adata, swap_latent=False)
rna_sample = generated_data[:,:hvg].copy()
proteins_sample = pd.DataFrame(generated_data[:,hvg:],  index= adata.obs_names, columns = adata.obsm['protein_counts'].columns)
adata.obsm['RNA_Z1_denoised'] = rna_sample
adata.obsm['protein_Z1_denoised'] = proteins_sample

generated_data = vae.posterior_predictive_sample(adata, swap_latent=True)
rna_sample = generated_data[:,:hvg].copy()
proteins_sample = pd.DataFrame(generated_data[:,hvg:],  index= adata.obs_names, columns = adata.obsm['protein_counts'].columns)
adata.obsm['RNA_Z2_denoised'] = rna_sample
adata.obsm['protein_Z2_denoised'] = proteins_sample

In [ ]:
vae.save("./outputs/Hao/saved_model/",  overwrite = True)

In [ ]:
adata.write("./outputs/Hao/multiHIVE.h5ad")